# Math Agent Demo

This notebook shows a small tool-using agent built from shared platform nodes.

It uses:
- `ReactNode` for planning
- `ToolExecutionNode`
- `ResponseNode`
- two reusable tools from `src.tools.math`

This React node carries its own tool catalog from `ToolRegistry.build_catalog_json()`.

The agent can:
- calculate arithmetic expressions
- convert between a small set of units

This example does not use memory.


In [1]:
from __future__ import annotations

import json
import re
import subprocess
import sys
from pathlib import Path
from types import SimpleNamespace


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current notebook path.")


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

required_packages = ["langgraph", "transformers", "accelerate", "sentencepiece"]
for package in required_packages:
    try:
        __import__(package)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

print(f"Using repository root: {repo_root}")
print(f"Python executable: {sys.executable}")


Using repository root: /Users/saketm10/Projects/openclaw_agents
Python executable: /Users/saketm10/Projects/openclaw_agents/.venv/bin/python3.11


In [2]:
from langgraph.graph import END, START, StateGraph
from src.agents.nodes import AgentState, ReactNode, ResponseNode, ToolExecutionNode
from src.llm.qwen import Qwen3_1_7BLLM
from src.tools.math import CalculateTool, UnitConvertTool
from src.tools.registry import ToolRegistry


qwen_llm = Qwen3_1_7BLLM(model_name="Qwen/Qwen3-1.7B", max_new_tokens=192, enable_thinking=False)

registry = ToolRegistry()
registry.register(CalculateTool())
registry.register(UnitConvertTool())
react_node = ReactNode(
    llm=qwen_llm,
    system_prompt=(
        "You are the planning node for a math agent. "
        "You will receive JSON containing the user_input and a structured tool_catalog_json string. "
        "Choose exactly one tool from the tool catalog and return only JSON in the form "
        "{\"tool_name\": \"...\", \"arguments\": {...}, \"thought\": \"...\"}."
    ),
    user_prompt=(
        "{\n"
        "  \"user_input\": \"{user_input}\",\n"
        "  \"tool_catalog_json\": {available_tools}\n"
        "}"
    ),
    available_tools=registry.build_catalog_json(),
)
tool_node = ToolExecutionNode(registry=registry)
response_node = ResponseNode(
    llm=qwen_llm,
    system_prompt="You are the response node for a math agent. Reply briefly and clearly.",
    user_prompt="User request: {user_input}\nTool output: {observation}",
    default_response="I could not complete that math request.",
)

graph = StateGraph(AgentState)
graph.add_node("react", react_node.execute)
graph.add_node("act", tool_node.execute)
graph.add_node("respond", response_node.execute)
graph.add_edge(START, "react")
graph.add_conditional_edges(
    "react",
    react_node.route,
    {"act": "act", "respond": "respond", "end": END},
)
graph.add_edge("act", "respond")
graph.add_edge("respond", END)
compiled_graph = graph.compile()


In [3]:
state = compiled_graph.invoke(
    {
        "user_input": "What is 12 * (3 + 4)?",
        "memory": None,
        "steps": 0,
    }
)
state["response"]


'The result of 12 * (3 + 4) is 84.'

In [4]:
state = compiled_graph.invoke(
    {
        "user_input": "Convert 5 miles to km.",
        "memory": None,
        "steps": 0,
    }
)
state["response"]


'5 miles is equal to 8.04672 kilometers.'

In [5]:
registry.build_catalog_json()

'[\n  {\n    "type": "function",\n    "function": {\n      "name": "calculate",\n      "description": "Evaluate a basic arithmetic expression.",\n      "parameters": {\n        "description": "Defines the validated input for arithmetic evaluation.",\n        "properties": {\n          "expression": {\n            "description": "Arithmetic expression such as 12 * (3 + 4).",\n            "title": "Expression",\n            "type": "string"\n          }\n        },\n        "required": [\n          "expression"\n        ],\n        "title": "CalculateInput",\n        "type": "object"\n      }\n    }\n  },\n  {\n    "type": "function",\n    "function": {\n      "name": "unit_convert",\n      "description": "Convert a numeric value between common length, weight, and temperature units.",\n      "parameters": {\n        "description": "Defines the validated input for unit conversion.",\n        "properties": {\n          "value": {\n            "description": "Numeric value to convert.",\n  